# 12. Análisis final basado en artefactos

Integrar datos, metodología y resultados leyendo archivos existentes, sin transcribir métricas manualmente.

**Dataset principal:** `data/nuevo_dataset/cardio_train.csv`. Este cuaderno no ejecuta entrenamientos. Los resultados históricos se leen de archivos conservados; no se sustituyen por nuevas ejecuciones.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/nuevo_dataset/cardio_train.csv").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from src.experimento import (
    ORIGINAL, RESULTADOS, VARIABLES, NOMBRES, cargar_datos, reglas_calidad,
    cargar_metricas, comparar_ejecuciones, tabla_configuracion, verificar_configuracion,
    construir_cnn, tabla_modelo, dibujar_arquitectura, guardar_figura_nueva,
    importancia_guardada, resumen_dataset, verificar_predicciones, tensorflow_disponible,
)

In [2]:
dataset = resumen_dataset()
comparacion = comparar_ejecuciones()
arquitectura = verificar_configuracion()
metricas = cargar_metricas("metricas_evaluacion.csv")
importancia = importancia_guardada()
verificadas, matriz = verificar_predicciones()
display(dataset)
display(comparacion)
display(arquitectura)
display(metricas)
display(importancia)
assert arquitectura.parametros.sum() == 34529

,Etapa,Registros,Columnas,Predictoras,Clase 0,Clase 1
0,Original,70000,13,11,35021,34979
1,Limpio,68615,13,11,34670,33945
2,Muestra,15000,13,11,7579,7421


,Protocolo,Entrenamiento,Validación,Prueba,Épocas,Parámetros,Accuracy,Precision,Recall,F1,ROC-AUC
0,StandardScaler,10500,2250,2250,44,34529,0.712444,0.716141,0.693621,0.704701,0.775738
1,MinMaxScaler,10200,2550,2250,66,34529,0.731111,0.732601,0.718778,0.725624,0.788458


,capa,tipo,salida,parametros,filtros,kernel,activacion,padding,unidades,dropout
0,conv1d,Conv1D,"(11, 32)",128,32.0,3.0,relu,same,NaN,NaN
1,conv1d_1,Conv1D,"(11, 96)",9312,96.0,3.0,relu,same,NaN,NaN
2,global_average_pooling1d,GlobalAveragePooling1D,"(96,)",0,NaN,NaN,—,—,NaN,NaN
3,dense,Dense,"(256,)",24832,NaN,NaN,relu,—,256.0,NaN
4,dropout,Dropout,"(256,)",0,NaN,NaN,—,—,NaN,0.3
5,dense_1,Dense,"(1,)",257,NaN,NaN,sigmoid,—,1.0,NaN


,Metrica,Valor
0,Accuracy,0.731111
1,Precision,0.732601
2,Recall,0.718778
3,F1-score,0.725624
4,ROC-AUC,0.788458


,variable,variable_es,importancia
0,ap_hi,Presión sistólica,0.747432
1,age,Edad,0.151009
2,cholesterol,Colesterol,0.074941
3,weight,Peso,0.010035
4,ap_lo,Presión diastólica,0.008521
5,height,Altura,0.003609
6,gluc,Glucosa,0.002322
7,active,Actividad física,0.001791
8,gender,Sexo,0.000341
9,smoke,Tabaquismo,0.000000


## Metodología y límites
El original se auditó mediante reglas operacionales; se filtró y se seleccionó una muestra estratificada ya existente. Se excluyó `id`, se convirtió edad a años y se mantuvieron 11 predictoras. La CNN se implementó con TensorFlow/Keras. Los dos protocolos usan la misma arquitectura.

La ejecución definitiva ajustó el escalador antes de reservar validación interna; la prueba quedó fuera del ajuste. Las pruebas de ambas ejecuciones no se obtuvieron con el mismo procedimiento. No hay estimación de variabilidad entre semillas ni validación externa. Las importancias corresponden a un árbol independiente. No se guardó el escalador definitivo ni una lista explícita de identificadores de las particiones. La secuencia documenta resultados existentes y no acredita una nueva evaluación del procedimiento recomendado sin fuga.

In [3]:
from IPython.display import Markdown
valores = metricas.set_index("Metrica").Valor
filas = dataset.set_index("Etapa").Registros
epocas = len(pd.read_csv(RESULTADOS / "tablas/historial_entrenamiento.csv"))
total = int(arquitectura.parametros.sum())
texto = (f"La muestra experimental contiene **{int(filas.iloc[-1]):,} registros** seleccionados de "
         f"{int(filas.iloc[0]):,}; el conjunto limpio contiene {int(filas.iloc[1]):,}. "
         f"El modelo utiliza {len(VARIABLES)} predictoras y **{total:,} parámetros**. "
         f"El entrenamiento definitivo ejecutó {epocas} épocas con restauración de mejores pesos. "
         + "Resultados conservados: " + "; ".join(f"{m}: **{v:.4f}**" for m, v in valores.items()) + ". "
         + f"La variable de mayor importancia en el árbol complementario es {importancia.iloc[0].variable_es}. "
         + "Estos resultados describen el protocolo ejecutado y deben interpretarse con las limitaciones documentadas.")
display(Markdown(texto))
print("Verificación de métricas desde predicciones y parámetros desde modelo: satisfactoria.")

Verificación de métricas desde predicciones y parámetros desde modelo: satisfactoria.


La muestra experimental contiene **15,000 registros** seleccionados de 70,000; el conjunto limpio contiene 68,615. El modelo utiliza 11 predictoras y **34,529 parámetros**. El entrenamiento definitivo ejecutó 66 épocas con restauración de mejores pesos. Resultados conservados: Accuracy: **0.7311**; Precision: **0.7326**; Recall: **0.7188**; F1-score: **0.7256**; ROC-AUC: **0.7885**. La variable de mayor importancia en el árbol complementario es Presión sistólica. Estos resultados describen el protocolo ejecutado y deben interpretarse con las limitaciones documentadas.